# Training the model the site uses

Sam Strickler · Honors research project · Home Credit Default Risk (Kaggle)

This notebook builds the model behind the preapproval website and saves it to `models/`.
Run it top to bottom, then restart the backend. It takes a few minutes on an M5 MacBook Pro
(longer with either of the reruns switched on at the top).

The model is a LightGBM classifier that only sees what the preapproval form asks. It leaves
out sex and marital status, which US lenders aren't allowed to use. 02_research.ipynb
explains both choices (section 3) and the training settings (section 2). It is trained to
cope with blank bureau scores, since most people filling in the form won't know theirs
(section 4).

The feature formulas live in `features.py` so the backend can import the same ones.

In [1]:
import json
import os

import joblib
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split

import features

SEED = 42
# The backend loads models/<MODEL_NAME>* (backend/app/paths.py), and so does the Dockerfile.
MODEL_NAME = "lightgbm_form_v4"
SITE_DATA = "../frontend/public/data"

# Rerun the hyperparameter search (about ten minutes on an M5). Off, training uses the
# setting the last search chose, saved in results/best_params.json.
RUN_SEARCH = False
# Rerun the comparison of how often to hide the bureau scores in training (section 4, four
# cross-validations). Off, training uses the share saved in results/score_hiding.json.
RUN_SCORE_SWEEP = False

/opt/miniconda3/envs/honorsproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the features

`features.py` builds about 1,570 features from the application and the five credit-history
tables and caches them in `cache/`. Most are dropped in section 3. The split is 80/20,
stratified, seed 42, the same one 02_research uses.

In [2]:
if os.path.exists(features.FULL_PATH) and os.path.exists(features.EXTRA_PATH):
    print("using the cached feature matrices in cache/")
else:
    features.build_cache()

X, y = features.load_matrix()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
del X, y
print(f"train {len(X_train):,}   test {len(X_test):,}   {X_train.shape[1]:,} features   "
      f"base rate {y_train.mean():.2%}")

using the cached feature matrices in cache/


train 246,005   test 61,502   1,573 features   base rate 8.07%


## 2. How a setting is scored

Every choice below is made with 5-fold cross-validation on the training split. The test set
is only used at the end, to report.

Each fold keeps adding trees until its held-out fifth stops improving for 200 rounds. Early
stopping has to watch AUC and nothing else. Left alone, LightGBM also watches log-loss and
stops far too early (02_research §2.4).

The same fifth both stops the training and scores it, so these cross-validation numbers run
a little high. The test set in section 7 is the clean check.

Each fold is scored twice: as it is, and with every applicant's bureau scores blanked, which
is how most applications reach the site. `hide_share` blanks the scores on a random share of
each fold's training rows (section 4 explains why); the held-out fifth is never touched by it.

In [3]:
LGB_BASE = dict(objective="binary", metric="auc", verbosity=-1, n_jobs=-1, random_state=SEED)


def built_from_scores(col):
    """Features that only exist when the bureau scores do. The cohort's own average score
    (COH_*_EXT_SOURCES_MEAN_MEAN) is not one: it comes from the cohort tables, not the applicant."""
    if col.startswith("EXT_SOURCE"):
        return col != "EXT_SOURCES_NAN_COUNT"
    return col.startswith("COH_") and col.endswith("_EXT_SOURCES_MEAN_Z")


def hide_scores(X, rows=None):
    """X with the bureau scores blanked on `rows` (all rows if None): exactly the inputs
    `features.add_application_features` builds when no score is given, as the backend does
    for a blank form. Two features aren't simply missing then: the count of missing scores
    is 3, and the weighted sum treats a missing score as 0."""
    X = X.copy()
    rows = np.ones(len(X), dtype=bool) if rows is None else rows
    X.loc[rows, [c for c in X.columns if built_from_scores(c)]] = np.nan
    if "EXT_SOURCES_NAN_COUNT" in X.columns:
        X.loc[rows, "EXT_SOURCES_NAN_COUNT"] = 3
    if "EXT_SOURCES_WEIGHTED" in X.columns:
        X.loc[rows, "EXT_SOURCES_WEIGHTED"] = 0.0
    return X


def stop_after(rounds):
    return [early_stopping(rounds, first_metric_only=True, verbose=False), log_evaluation(0)]


def cross_validate(params, X, y, n_folds=5, hide_share=0.0):
    """Out-of-fold predictions with and without bureau scores, per-fold AUC and tree count,
    and mean feature gain."""
    folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof, oof_blank = np.zeros(len(X)), np.zeros(len(X))
    fold_aucs, trees, gain = [], [], np.zeros(X.shape[1])
    for fold, (fit_idx, val_idx) in enumerate(folds.split(X, y)):
        hidden = np.random.default_rng(SEED + fold).random(len(fit_idx)) < hide_share
        model = LGBMClassifier(**LGB_BASE, **params, n_estimators=20_000)
        model.fit(hide_scores(X.iloc[fit_idx], hidden), y.iloc[fit_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], eval_metric="auc",
                  callbacks=stop_after(200))
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        oof_blank[val_idx] = model.predict_proba(hide_scores(X.iloc[val_idx]))[:, 1]
        fold_aucs.append(roc_auc_score(y.iloc[val_idx], oof[val_idx]))
        trees.append(model.best_iteration_)
        gain += model.booster_.feature_importance("gain") / n_folds
        print(f"  fold {fold + 1}/{n_folds}  AUC {fold_aucs[-1]:.5f}  "
              f"scores blank {roc_auc_score(y.iloc[val_idx], oof_blank[val_idx]):.5f}  {trees[-1]:,} trees")
    return {"cv_auc": roc_auc_score(y, oof), "cv_auc_blank": roc_auc_score(y, oof_blank),
            "fold_aucs": fold_aucs, "trees": trees, "oof": oof, "oof_blank": oof_blank, "gain": gain}

## 3. Keep only what a form can ask

`features.form_feature_columns` keeps the features a filled-in form can produce: the loan
amounts and their ratios, age, household, education, job, and the optional bureau scores and
card utilization.

Sex and marital status are left out, along with the cohort features built on sex. That cost
about 0.002 AUC (02_research §3.1). Both are kept for the test rows only, to check approval
rates by group in section 9.

In [4]:
form_cols = features.form_feature_columns(X_train.columns)
protected = [c for c in features.form_feature_columns(X_train.columns, include_protected=True)
             if c not in set(form_cols)]
# Never a model input: only used to check approval rates by group in section 9
groups_test = X_test[features.PROTECTED].astype(str)

X_train, X_test = X_train[form_cols], X_test[form_cols]
print(f"{len(form_cols)} features a form can supply")
print(f"left out: {', '.join(features.PROTECTED)} and {len(protected) - len(features.PROTECTED)} cohort "
      "features built on sex")

105 features a form can supply
left out: CODE_GENDER, NAME_FAMILY_STATUS and 16 cohort features built on sex


### Better bureau scores can only lower the risk

A tree model can do odd things where the data is thin. An early version of this model
predicted 2.4% risk with all three bureau scores at 0.95 and 2.1% at 0.8, so a better score
raised the risk.

LightGBM's monotone constraints rule that out. Every feature that goes up with the bureau
scores is only allowed to push the risk down.

The spread between the three scores is dropped. It can move either way when one score
improves, and it made three equal scores look risky, which is rare in the data but easy to
hit on a slider.

In [5]:
X_train, X_test = X_train.drop(columns="EXT_SOURCES_STD"), X_test.drop(columns="EXT_SOURCES_STD")
# With the spread gone, every feature built from the scores rises with them
constrained = [c for c in X_train.columns if built_from_scores(c)]
LGB_BASE.update(
    monotone_constraints=[-1 if built_from_scores(c) else 0 for c in X_train.columns],
    # "advanced" enforces the same guarantee with less damage to accuracy than the default
    monotone_constraints_method="advanced",
)
print(f"{X_train.shape[1]} features after dropping the score spread; "
      f"{len(constrained)} constrained so that higher never means riskier:")
print(", ".join(constrained))

104 features after dropping the score spread; 18 constrained so that higher never means riskier:
EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3, EXT_SOURCES_MEAN, EXT_SOURCES_MIN, EXT_SOURCES_MAX, EXT_SOURCES_PROD, EXT_SOURCES_WEIGHTED, EXT_SOURCE_1_X_AGE, EXT_SOURCE_1_X_EMPLOYED, EXT_SOURCE_2_X_AGE, EXT_SOURCE_2_X_EMPLOYED, EXT_SOURCE_3_X_AGE, EXT_SOURCE_3_X_EMPLOYED, COH_ORG_EXT_SOURCES_MEAN_Z, COH_OCC_EXT_SOURCES_MEAN_Z, COH_EDU_EXT_SOURCES_MEAN_Z, COH_INC_EXT_SOURCES_MEAN_Z


## 4. Hyperparameter search

Optuna tries 60 settings. Each one is trained on 75% of the training split and scored on the
other 25%, with a faster learning rate to keep trials short. One split is a noisy judge
(02_research §2.7), so the five best are re-scored with full cross-validation and the best
of those wins. Early stopping picks the number of trees.

The search is off by default and reuses the winner saved in `results/best_params.json`. Set
`RUN_SEARCH = True` at the top to run it again (about ten minutes).

In [6]:
SEARCH_LEARNING_RATE = 0.05
FINAL_LEARNING_RATE = 0.012


def suggest(trial):
    return {
        "num_leaves": trial.suggest_int("num_leaves", 16, 96),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 0.75),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [127, 255, 511]),
        "cat_smooth": trial.suggest_int("cat_smooth", 1, 100),
    }


if RUN_SEARCH:
    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                                  random_state=SEED, stratify=y_train)

    def objective(trial):
        model = LGBMClassifier(**{**LGB_BASE, "n_jobs": 5}, **suggest(trial), subsample_freq=1,
                               learning_rate=SEARCH_LEARNING_RATE, n_estimators=20_000)
        model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], eval_metric="auc", callbacks=stop_after(100))
        return model.best_score_["valid_0"]["auc"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    # Two trials at once, five threads each. On an M5 that gets about 30% more done than one
    # trial using every core: memory bandwidth, not core count, is what limits LightGBM.
    study.optimize(objective, n_trials=60, n_jobs=2, show_progress_bar=True)

    finalists = sorted((t for t in study.trials if t.value is not None),
                       key=lambda t: t.value, reverse=True)[:5]
    rescored = []
    for t in finalists:
        candidate = {**t.params, "subsample_freq": 1, "learning_rate": FINAL_LEARNING_RATE}
        print(f"trial {t.number}: holdout AUC {t.value:.5f}")
        rescored.append((cross_validate(candidate, X_train, y_train)["cv_auc"], candidate))
    cv_auc, winner = max(rescored, key=lambda r: r[0])
    with open("results/best_params.json", "w") as f:
        json.dump({"params": winner, "cv_auc": cv_auc}, f, indent=2)
    print(f"\nwinner: 5-fold CV AUC {cv_auc:.5f}")

params = json.load(open("results/best_params.json"))["params"]
pd.Series(params).to_frame("value")

,value
num_leaves,27.000000
max_depth,7.000000
min_child_samples,163.000000
min_child_weight,4.276984
subsample,0.835818
colsample_bytree,0.172768
reg_alpha,1.677578
reg_lambda,0.004522
max_bin,255.000000
cat_smooth,70.000000


### Training for blank bureau scores

Almost every applicant in the data has a bureau score; only 0.06% have none. Most people
filling in the form don't know theirs and leave them blank. A model trained only on
applicants with scores leans on them heavily and has never learned what to do without them.

So the scores, and everything built from them, are hidden on a random share of the training
rows. The model then learns both cases: how to use the scores when it has them, and how to
rank applicants on the rest of the form when it doesn't. The site sends a blank form as
missing scores, the same as these hidden rows.

Hiding more helps the blank case and costs the case with scores. The rule, set before the
comparison was run: take the share with the best blank-score CV AUC among those that cost
the scores-given CV AUC no more than 0.002 against hiding nothing (about half the test set's
standard error). The hyperparameters stay the ones the search chose without hiding.

In [7]:
HIDE_SHARES = [0.0, 0.25, 0.5, 0.75]
MAX_COST_WITH_SCORES = 0.002

if RUN_SCORE_SWEEP or not os.path.exists("results/score_hiding.json"):
    sweep = []
    for share in HIDE_SHARES:
        print(f"scores hidden on {share:.0%} of training rows")
        r = cross_validate(params, X_train, y_train, hide_share=share)
        sweep.append({"hidden share": share, "CV AUC": r["cv_auc"],
                      "CV AUC, scores blank": r["cv_auc_blank"], "trees": int(np.mean(r["trees"]))})
    with open("results/score_hiding.json", "w") as f:
        json.dump(sweep, f, indent=2)

sweep = pd.DataFrame(json.load(open("results/score_hiding.json")))
no_hiding = sweep.loc[sweep["hidden share"] == 0, "CV AUC"].iloc[0]
allowed = sweep[sweep["CV AUC"] >= no_hiding - MAX_COST_WITH_SCORES]
HIDE_SHARE = float(allowed.loc[allowed["CV AUC, scores blank"].idxmax(), "hidden share"])
print(f"chosen: hide the scores on {HIDE_SHARE:.0%} of training rows")
sweep.round(5)

scores hidden on 0% of training rows


  fold 1/5  AUC 0.76367  scores blank 0.69436  1,776 trees


  fold 2/5  AUC 0.76717  scores blank 0.69592  2,194 trees


  fold 3/5  AUC 0.76279  scores blank 0.68693  1,990 trees


  fold 4/5  AUC 0.76950  scores blank 0.70084  2,061 trees


  fold 5/5  AUC 0.76068  scores blank 0.68513  1,378 trees
scores hidden on 25% of training rows


... (12 outputs left out)

  fold 3/5  AUC 0.76131  scores blank 0.69982  2,034 trees


  fold 4/5  AUC 0.76872  scores blank 0.71198  1,993 trees


  fold 5/5  AUC 0.76019  scores blank 0.69571  1,392 trees
chosen: hide the scores on 75% of training rows


,hidden share,CV AUC,"CV AUC, scores blank",trees
0,0.00,0.76474,0.67263,1879
1,0.25,0.76432,0.70017,1683
2,0.50,0.76435,0.70228,1657
3,0.75,0.76399,0.70361,1771


## 5. Cross-validate the chosen setting

This gives the number of trees for the final model (the folds' average) and the out-of-fold
predictions section 8 uses to pick the cutoff, with the scores hidden on the share chosen
above.

In [8]:
cv = cross_validate(params, X_train, y_train, hide_share=HIDE_SHARE)
n_trees = int(np.mean(cv["trees"]))
print(f"\n5-fold CV AUC {cv['cv_auc']:.5f}   (spread between folds {np.std(cv['fold_aucs']):.5f})")
print(f"5-fold CV AUC with the scores blank {cv['cv_auc_blank']:.5f}")
print(f"final model: {n_trees:,} trees")

  fold 1/5  AUC 0.76407  scores blank 0.70186  1,791 trees


  fold 2/5  AUC 0.76606  scores blank 0.70878  1,647 trees


  fold 3/5  AUC 0.76131  scores blank 0.69982  2,034 trees


  fold 4/5  AUC 0.76872  scores blank 0.71198  1,993 trees


  fold 5/5  AUC 0.76019  scores blank 0.69571  1,392 trees

5-fold CV AUC 0.76399   (spread between folds 0.00311)
5-fold CV AUC with the scores blank 0.70361
final model: 1,771 trees


## 6. Train the model

The chosen setting, fit on the whole training split, with the scores hidden on the same
share of rows. There is no class weighting: it added no AUC and distorted the probabilities
(02_research §2.6).

In [9]:
hidden = np.random.default_rng(SEED).random(len(X_train)) < HIDE_SHARE
model = LGBMClassifier(**LGB_BASE, **params, n_estimators=n_trees)
model.fit(hide_scores(X_train, hidden), y_train)
print(f"trained {n_trees:,} trees on {len(X_train):,} applications, "
      f"{hidden.sum():,} of them with the scores hidden")

trained 1,771 trees on 246,005 applications, 184,397 of them with the scores hidden


## 7. Evaluate on the test set

- **AUC**: the chance the model ranks someone who missed a payment above someone who didn't.
- **Gini**: 2 × AUC − 1, the same thing on the scale lenders use.
- **KS**: the biggest gap between how the two groups are scored.
- **Lift**: average precision divided by the 8% base rate.
- **Calibration**: when the model says 10%, do about 10% miss a payment?
- **Gains**: turn away the riskiest N% and see how many missed payments that avoids.

With about 5,000 missed payments in the test set, the AUC is good to about ±0.004
(02_research §2.1).

In [10]:
proba = model.predict_proba(X_test)[:, 1]
actual = y_test.to_numpy()
base_rate = actual.mean()

auc = roc_auc_score(actual, proba)
ap = average_precision_score(actual, proba)
fpr, tpr, _ = roc_curve(actual, proba)
ks = float(np.max(tpr - fpr))

metrics = {"auc": auc, "gini": 2 * auc - 1, "ks": ks, "ap": ap, "ap_lift": ap / base_rate,
           "base_rate": base_rate, "brier": brier_score_loss(actual, proba),
           "mean_predicted": proba.mean()}
pd.Series(metrics).round(4).to_frame("test set")

,test set
auc,0.7669
gini,0.5337
ks,0.3984
ap,0.2523
ap_lift,3.1255
base_rate,0.0807
brier,0.0673
mean_predicted,0.0817


In [11]:
deciles = pd.DataFrame({"predicted": proba, "actual": actual})
deciles["tenth"] = pd.qcut(deciles["predicted"], 10, labels=False) + 1
deciles.groupby("tenth").agg(predicted=("predicted", "mean"), actual=("actual", "mean"),
                             applications=("actual", "size")).round(4)

,predicted,actual,applications
tenth,,,
1,0.0118,0.0124,6151
2,0.0196,0.0185,6150
3,0.0264,0.0239,6150
4,0.0341,0.0350,6150
5,0.0438,0.0459,6150
6,0.0567,0.0571,6150
7,0.0753,0.0790,6150
8,0.1037,0.1067,6150
9,0.1533,0.1468,6150


In [12]:
riskiest_first = actual[np.argsort(-proba)]
rows = []
for pct in (5, 10, 20, 30, 40, 50):
    n = int(len(actual) * pct / 100)
    caught = riskiest_first[:n].sum()
    rows.append({"decline riskiest": f"{pct}%",
                 "missed payments avoided": f"{caught / actual.sum():.1%}",
                 "miss rate in that group": f"{caught / n:.1%}",
                 "good applicants turned away": f"{n - caught:,}"})
pd.DataFrame(rows)

,decline riskiest,missed payments avoided,miss rate in that group,good applicants turned away
0,5%,21.5%,34.7%,"2,007"
1,10%,34.9%,28.2%,"4,416"
2,20%,53.1%,21.4%,"9,662"
3,30%,66.3%,17.9%,"15,156"
4,40%,76.1%,15.4%,"20,820"
5,50%,83.2%,13.4%,"26,620"


### Without bureau scores

Every test applicant has bureau scores on file. Someone filling in the form usually won't
know theirs, and the site then sends them as missing. This rebuilds the test set from the
raw form answers both ways, the way the backend does. The first row should match the AUC
above; the second is what the model can do when the scores are blank.

In [13]:
scores = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]


def rebuilt_test_set(blank_scores):
    """The test set's model inputs, rebuilt from the raw form answers the way the backend does it."""
    raw = pd.read_parquet(features.FULL_PATH, columns=["TARGET"] + features.FORM_INPUTS)
    in_train = features.train_mask(raw["TARGET"])
    if blank_scores:
        raw.loc[~in_train, scores] = np.nan
    raw = features.add_application_features(raw)
    rebuilt = pd.concat([raw, features.cohort_features(raw, in_train)], axis=1).loc[X_test.index, X_test.columns]
    for col in X_test.select_dtypes("category"):
        rebuilt[col] = pd.Categorical(rebuilt[col], categories=X_test[col].cat.categories)
    return rebuilt


auc_rebuilt = roc_auc_score(actual, model.predict_proba(rebuilt_test_set(blank_scores=False))[:, 1])
proba_blank = model.predict_proba(rebuilt_test_set(blank_scores=True))[:, 1]
metrics.update(auc_scores_blank=roc_auc_score(actual, proba_blank),
               ap_scores_blank=average_precision_score(actual, proba_blank),
               mean_predicted_scores_blank=proba_blank.mean())
print(f"AUC with bureau scores on file:    {auc_rebuilt:.4f}")
print(f"AUC with bureau scores left blank: {metrics['auc_scores_blank']:.4f}   "
      f"(average precision {metrics['ap_scores_blank']:.4f}, mean prediction {proba_blank.mean():.2%})")

AUC with bureau scores on file:    0.7669
AUC with bureau scores left blank: 0.7060   (average precision 0.1851, mean prediction 8.02%)


## 8. Choose the cutoff

Applications that score above the cutoff are declined. The cutoff is the score with the best
F1 (the balance of precision and recall) on the out-of-fold predictions from section 5, so
the test set stays out of the choice.

F1 is a stand-in. A real lender would set the cutoff from what a missed payment costs
against what a good loan earns.

In [14]:
precision, recall, cutoffs = precision_recall_curve(y_train, cv["oof"])
f1 = 2 * precision * recall / (precision + recall + 1e-12)
threshold = float(cutoffs[f1[:-1].argmax()])
print(f"cutoff {threshold:.4f}  (F1 {f1[:-1].max():.3f} out of fold)")

for label, suffix, p in (("scores on file", "", proba), ("scores left blank", "_scores_blank", proba_blank)):
    tn, fp, fn, tp = confusion_matrix(actual, p >= threshold).ravel()
    metrics[f"declined{suffix}"], metrics[f"caught{suffix}"] = (tp + fp) / len(p), tp / (tp + fn)
    print(f"test set, {label}: declines {(tp + fp) / len(p):.1%} of applicants and catches "
          f"{tp / (tp + fn):.1%} of the missed payments; miss rate among those approved "
          f"{fn / (tn + fn):.2%}, against {base_rate:.2%} if everyone were")

cutoff 0.1411  (F1 0.311 out of fold)
test set, scores on file: declines 16.5% of applicants and catches 47.9% of the missed payments; miss rate among those approved 5.04%, against 8.07% if everyone were
test set, scores left blank: declines 12.5% of applicants and catches 32.4% of the missed payments; miss rate among those approved 6.23%, against 8.07% if everyone were


## 9. Approval rates by sex and marital status

The model never sees either one, but other features can carry some of the same signal, so
the outcomes are worth checking. Each row shows how often a group is approved, the risk the
model predicts for it on average, and how often it actually missed a payment. If the last
two columns agree, the model isn't over- or under-stating that group's risk.

In [15]:
outcome = groups_test.assign(approved=proba < threshold, predicted=proba, missed=actual)
for col, label in (("CODE_GENDER", "sex"), ("NAME_FAMILY_STATUS", "marital status")):
    table = outcome.groupby(col).agg(applicants=("approved", "size"), approved=("approved", "mean"),
                                     predicted_risk=("predicted", "mean"), missed_a_payment=("missed", "mean"))
    display(table.rename_axis(label).round(3))
rates = outcome.groupby("CODE_GENDER")["approved"].mean()
print(f"men are approved at {rates['M'] / rates['F']:.2f} times the rate of women")

,applicants,approved,predicted_risk,missed_a_payment
sex,,,,
F,40544,0.868,0.072,0.070
M,20958,0.770,0.100,0.102


,applicants,approved,predicted_risk,missed_a_payment
marital status,,,,
Civil marriage,6003,0.790,0.094,0.096
Married,39323,0.849,0.078,0.077
Separated,4015,0.830,0.083,0.088
Single / not married,8951,0.777,0.097,0.094
Unknown,1,1.000,0.028,0.000
Widow,3209,0.915,0.061,0.057


men are approved at 0.89 times the rate of women


## 10. Save the model

Everything the backend loads, in `models/`:

| file | what it is |
|---|---|
| `.joblib` | the trained model |
| `_threshold.json` | the cutoff from section 8 |
| `_percentiles.json` | where a score ranks among the test applicants |
| `_schema.json` | the feature order and category levels |
| `_cohorts.json` | cohort averages, so one applicant can be compared with their group |

LightGBM matches categories by position, so the schema keeps their order. Serve them in a
different order and "Laborers" would be read as some other occupation.

In [16]:
os.makedirs("models", exist_ok=True)
path = f"models/{MODEL_NAME}"

joblib.dump(model, f"{path}.joblib")

with open(f"{path}_threshold.json", "w") as f:
    json.dump({"threshold": threshold, "source": "out-of-fold F1-optimal"}, f, indent=2)

with open(f"{path}_percentiles.json", "w") as f:
    json.dump({"percentile_edges": [float(x) for x in np.percentile(proba, np.arange(1, 100))],
               "reference": "held-out test set", "n": int(len(proba))}, f)

categoricals = X_train.select_dtypes(include=["category"]).columns
with open(f"{path}_schema.json", "w") as f:
    json.dump({"features": list(X_train.columns),
               "categoricals": {c: [str(v) for v in X_train[c].cat.categories] for c in categoricals},
               "excluded": features.PROTECTED,
               "scores_hidden_share": HIDE_SHARE,
               "test_auc": float(auc), "test_auc_scores_blank": float(metrics["auc_scores_blank"]),
               "test_avg_precision": float(ap), "n_estimators": n_trees}, f, indent=2)

# Only the cohorts the model uses, from the training rows only: the same ones it learned from
cohorts_used = [keys for keys in features.COHORTS if set(keys) <= set(features.FORM_INPUTS)]
cohort_cols = ["SK_ID_CURR", "TARGET"] + sorted({c for keys in cohorts_used for c in keys}
                                                 | set(features.COHORT_NUMERICS))
cohorts = pd.read_parquet(features.FULL_PATH, columns=cohort_cols)
features.export_cohort_tables(cohorts, features.train_mask(cohorts["TARGET"]), f"{path}_cohorts.json",
                              cohorts=cohorts_used)

with open("results/final_metrics.json", "w") as f:
    json.dump({k: float(v) for k, v in {**metrics, "threshold": threshold}.items()}, f, indent=2)

for suffix in (".joblib", "_threshold.json", "_percentiles.json", "_schema.json"):
    print(f"  saved {path}{suffix}  ({os.path.getsize(path + suffix) / 1e6:.1f} MB)")

  wrote models/lightgbm_form_v4_cohorts.json: 4 cohorts, 90 groups, 43 KB
  saved models/lightgbm_form_v4.joblib  (5.5 MB)
  saved models/lightgbm_form_v4_threshold.json  (0.0 MB)
  saved models/lightgbm_form_v4_percentiles.json  (0.0 MB)
  saved models/lightgbm_form_v4_schema.json  (0.0 MB)


In [17]:
# The "where this application falls" chart on the site
CAP, BINS = 0.30, 30
counts, edges = np.histogram(np.clip(proba, 0, CAP), bins=BINS, range=(0, CAP))
with open(f"{SITE_DATA}/risk_distribution.json", "w") as f:
    json.dump({"counts": [int(c) for c in counts], "edges": [float(e) for e in edges],
               "cap": CAP, "n": int(len(proba)), "median": float(np.median(proba)),
               "share_in_range": float((proba < CAP).mean()),
               "tail_n": int((proba >= CAP).sum())}, f, indent=1)
print(f"  saved {SITE_DATA}/risk_distribution.json  (median score {np.median(proba):.4f})")

  saved ../frontend/public/data/risk_distribution.json  (median score 0.0494)


**Restart the backend after running this.** It loads the model once at startup.

## 11. Which form fields matter

Change one field at a time on a typical applicant and see how far the prediction moves. The
table is saved to `results/form_field_sensitivity.csv`.

In [18]:
schema = json.load(open(f"models/{MODEL_NAME}_schema.json"))


def as_model_input(application):
    """One application dict to a model row, the way the backend builds one (minus the
    cohort features, which need the backend's lookup tables)."""
    row = features.add_application_features(pd.DataFrame([application])).reindex(columns=schema["features"])
    for col, levels in schema["categoricals"].items():
        row[col] = pd.Categorical(row[col].astype("object"), categories=levels)
    for col in row.columns:
        if col not in schema["categoricals"]:
            row[col] = pd.to_numeric(row[col], errors="coerce").astype("float32")
    return row


typical = {
    "AMT_INCOME_TOTAL": 150000, "AMT_CREDIT": 500000, "AMT_ANNUITY": 25000,
    "AMT_GOODS_PRICE": 450000, "DAYS_BIRTH": -35 * 365, "DAYS_EMPLOYED": -5 * 365,
    "DAYS_EMPLOYED_ANOM": 0, "CNT_CHILDREN": 1, "CNT_FAM_MEMBERS": 3,
    "NAME_EDUCATION_TYPE": "Secondary / secondary special", "NAME_INCOME_TYPE": "Working",
    "OCCUPATION_TYPE": "Laborers", "EXT_SOURCE_1": 0.5, "EXT_SOURCE_2": 0.5, "EXT_SOURCE_3": 0.5,
}
alternatives = {
    "AMT_INCOME_TOTAL": [60000, 400000], "AMT_CREDIT": [150000, 1200000],
    "AMT_ANNUITY": [10000, 60000], "AMT_GOODS_PRICE": [150000, 1000000],
    "DAYS_BIRTH": [-22 * 365, -60 * 365], "DAYS_EMPLOYED": [-0.5 * 365, -25 * 365],
    "DAYS_EMPLOYED_ANOM": [1], "CNT_CHILDREN": [0, 4], "CNT_FAM_MEMBERS": [1, 6],
    "NAME_EDUCATION_TYPE": ["Higher education", "Lower secondary"],
    "NAME_INCOME_TYPE": ["Pensioner", "Commercial associate"],
    "OCCUPATION_TYPE": ["Managers", "Low-skill Laborers"],
    "EXT_SOURCE_1": [0.1, 0.9], "EXT_SOURCE_2": [0.1, 0.9], "EXT_SOURCE_3": [0.1, 0.9],
}

base_row = as_model_input(typical)
base_p = model.predict_proba(base_row)[0, 1]
rows = []
for field, values in alternatives.items():
    moved, inputs_changed = 0.0, 0
    for value in values:
        row = as_model_input({**typical, field: value})
        moved = max(moved, abs(model.predict_proba(row)[0, 1] - base_p))
        same = (row.iloc[0] == base_row.iloc[0]) | (row.iloc[0].isna() & base_row.iloc[0].isna())
        inputs_changed = max(inputs_changed, int((~same).sum()))
    rows.append({"form field": field, "model inputs changed": inputs_changed,
                 "largest move (points)": round(moved * 100, 3)})

print(f"typical applicant: {base_p:.2%} chance of an early missed payment")
sensitivity = pd.DataFrame(rows).sort_values("largest move (points)", ascending=False).reset_index(drop=True)
sensitivity.to_csv("results/form_field_sensitivity.csv", index=False)
sensitivity

typical applicant: 4.15% chance of an early missed payment


,form field,model inputs changed,largest move (points)
0,EXT_SOURCE_3,7,4.416
1,EXT_SOURCE_1,7,3.263
2,EXT_SOURCE_2,7,2.304
3,AMT_ANNUITY,3,2.204
4,AMT_GOODS_PRICE,4,2.074
5,CNT_FAM_MEMBERS,3,1.307
6,CNT_CHILDREN,3,0.639
7,DAYS_EMPLOYED,5,0.613
8,AMT_CREDIT,5,0.580
9,AMT_INCOME_TOTAL,6,0.520


## 12. What these numbers don't cover

- **Later applicants.** The split is random because the data has no application dates, so
  nothing here shows how the model holds up over time.
- **People a lender turns away.** Everyone in the data was approved by Home Credit. The
  model has never seen a rejected applicant.
- **Losses.** The target is one early late payment. Most of those borrowers go on to repay.
- **Household size.** The form still asks for children and people in the household, which
  together hint at whether someone has a partner.